# Composite figure — six panels, paper-ready

Reads the saved CSVs from the analysis notebooks and renders six panels:

- **(a)** Test $R^{2}$ vs M (Ridge sweep) — reference clocks as **dashed segments with arrows** to a label column on the right (no inline-label overlap).
- **(b)** Held-out test scatter at M = 100 (Ridge), solid fit line.
- **(c)** External-cohort scatter (Ridge clock), solid fit line.
- **(d)** PCA $R^{2}$ vs K on external cohorts, **same arrow style as (a)**.
- **(e)** External-cohort scatter (PCA clock), solid fit line.
- **(f)** Pooled MAE box plot combining the PCA clock, the Ridge "Network Clock", and the five reference clocks — order set by `BOX_ORDER`.

A single `CLOCK_COLORS` dict at the top fixes the colour of every clock once and for all — so the same clock looks identical in (a), (d), and (f) regardless of which other clocks happen to be plotted alongside.

*Naming convention*: we never say "clusters" — always **modules**.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.metrics import r2_score, mean_absolute_error


# ── Paths to the CSVs written by the analysis notebooks ─────────────
CLOCK_DIR   = Path("../outputs/05_clock/random")
SUMMARY_CSV = Path("../outputs/05_clock/all_modes_summary.csv")
TEST_DIR    = Path("../outputs/test_network_clock_3cohorts")
OUT_DIR     = Path("../outputs/composite_figure")
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ── Box-plot order in panel (f). Keys are CSV column keys, NOT pretty labels.
#    Set to None to auto-sort by mean error.
BOX_ORDER = None   # None → auto-sort boxes by ascending mean MAE


# ── Paper-ready style ───────────────────────────────────────────────
DPI = 300
PANEL_SIZE = 6.0
COMPOSITE_FIGSIZE = (3 * PANEL_SIZE + 1.5, 2 * PANEL_SIZE + 1.0)

mpl.rcParams.update({
    "font.family":       ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size":          13,
    "axes.labelsize":     14,
    "axes.titlesize":     14,
    "xtick.labelsize":    12,
    "ytick.labelsize":    12,
    "legend.fontsize":     9,
    "legend.frameon":     False,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.spines.left":   True,
    "axes.spines.bottom": True,
    "axes.edgecolor":     "black",
    "axes.linewidth":     1.4,
    "xtick.direction":    "out",
    "ytick.direction":    "out",
    "xtick.major.size":   5,
    "ytick.major.size":   5,
    "xtick.major.width":  1.2,
    "ytick.major.width":  1.2,
    "figure.dpi":         DPI,
    "savefig.dpi":        DPI,
    "savefig.bbox":       "tight",
    "figure.facecolor":   "white",
    "axes.facecolor":     "white",
    "axes.grid":          False,
})


# ── Colours ─────────────────────────────────────────────────────────
# SINGLE source of truth: every clock has one colour, used in every panel.
CLOCK_COLORS = {
    # Reference clocks
    "horvath2013":  "#999933",   # olive
    "hannum":       "#CC6677",   # rose
    "altumage":     "#117733",   # dark green
    "skinandblood": "#882255",   # plum
    "han":          "#88CCEE",   # sky blue
    # Our clocks
    "pca":          "#332288",   # deep indigo — PCA on modules
    "pca_network":  "#DDAA33",   # gold        — PCA on whole network
    "pca_array":    "#0077BB",   # vivid blue  — PCA on whole dataset
    "ridge":        "#EE7733",   # orange      — Ridge "Network Clock"
}
CLOCK_LABELS = {
    "horvath2013":  "Horvath",
    "hannum":       "Hannum",
    "altumage":     "AltumAge",
    "skinandblood": "Skin & Blood",
    "han":          "Han",
    "pca":          "PCA (modules)",
    "pca_network":  "PCA (whole network)",
    "pca_array":    "PCA (whole dataset)",
    "ridge":        "Network Clock",
}

# Short labels for panel (d) legend (the title "PCA clocks" carries the common prefix)
PCA_SHORT_LABELS = {
    "pca":          "modules",
    "pca_network":  "whole network",
    "pca_array":    "whole dataset",
}
REFERENCE_KEYS = ["horvath2013", "hannum", "altumage", "skinandblood", "han"]

# Sweep methods in panel (a) — neutral colours, these are NOT "clocks"
METHOD_COLOR = {"random_network": "#888888", "random_per_module": "#DDCC77"}
METHOD_LABEL = {"random_network": "Random CpGs from network",
                "random_per_module": "Random 1 per module"}

# Per-study palette in panel (b) (training cohorts, not clocks)
STUDY_PALETTE = ["#4477AA", "#EE6677", "#228833", "#CCBB44", "#66CCEE",
                 "#AA3377", "#BBBBBB", "#E69F00", "#56B4E9", "#009E73",
                 "#D55E00", "#CC79A7", "#0072B2", "#F0E442", "#882255"]

# Group / dataset markers in (c) and (e)
GROUP_COLORS = {
    "Control":              "#4477AA",
    "HIV":                  "#EE6677",
    "psoriasis_vulgaris":   "#CCBB44",
    "psoriasis_arthritis":  "#AA3377",
    "Aging study":          "#888888",
    "unknown":              "#BBBBBB",
}
DATASET_MARKERS = {"GSE235717": "o", "GSE217633": "s", "GSE200376": "^"}


## Load all CSVs

In [ ]:
# (a) Ridge sweep + reference clock summary (training-test fold)
df_sweep = pd.read_csv(CLOCK_DIR / "random_sweep_results.csv")
# Old method names may say "random_per_cluster" — alias to "random_per_module" for display
df_sweep["method"] = df_sweep["method"].replace({"random_per_cluster": "random_per_module"})

df_sum   = pd.read_csv(SUMMARY_CSV)
if "mode" in df_sum.columns:
    df_sum = df_sum[df_sum["mode"] == "random"]

# (b) Internal M=100 scatter
df_scatter_internal = pd.read_csv(CLOCK_DIR / "scatter_rand_clust_M100.csv")

# (c) External-cohort scatter for the Ridge clock
df_scatter_external_ridge = pd.read_csv(
    TEST_DIR / "scatter_random_per_cluster_ext_best.csv")

# (d) PCA R² vs K on external cohorts — three variants: modules, whole network, whole dataset
def _try_csv(p):
    try:    return pd.read_csv(p)
    except FileNotFoundError: return None

df_perf_pca         = pd.read_csv(TEST_DIR / "pca_clock_perf_by_K.csv")
df_perf_pca_network = _try_csv(TEST_DIR / "pca_network_clock_perf_by_K.csv")
df_perf_pca_array   = _try_csv(TEST_DIR / "pca_array_clock_perf_by_K.csv")
for nm, df in [("pca_network_clock_perf_by_K.csv", df_perf_pca_network),
               ("pca_array_clock_perf_by_K.csv",   df_perf_pca_array)]:
    if df is None:
        print(f"  ⚠ {nm} not found — that PCA curve will be skipped in (d)")

# (e) External-cohort scatter for the PCA clock (on modules)
df_scatter_external_pca = pd.read_csv(TEST_DIR / "scatter_pca_ext_best.csv")

# (f) Per-sample predictions — both Ridge and PCA CSVs
df_pred_ridge = pd.read_csv(TEST_DIR / "per_sample_predictions_ridge_ext_best.csv")
df_pred_pca   = pd.read_csv(TEST_DIR / "per_sample_predictions_pca_ext_best.csv")


# Reference clock pooled R² on the EXTERNAL cohorts (for the dashed baselines in (d))
def _age_col(df): return next((c for c in ["age", "Age"] if c in df.columns), None)
age_col_pca = _age_col(df_pred_pca)
ref_pooled_external = {}
for ref in REFERENCE_KEYS:
    if ref in df_pred_pca.columns and age_col_pca:
        sub = df_pred_pca[[age_col_pca, ref]].dropna()
        if len(sub) >= 2:
            ref_pooled_external[ref] = float(r2_score(sub[age_col_pca], sub[ref]))

print(f"  sweep:    {df_sweep.shape},  methods: {sorted(df_sweep['method'].unique())}")
print(f"  summary:  {df_sum.shape}")
print(f"  scat-int: {df_scatter_internal.shape}, cols: {list(df_scatter_internal.columns)}")
print(f"  scat-ext (ridge): {df_scatter_external_ridge.shape}")
print(f"  scat-ext (pca):   {df_scatter_external_pca.shape}")
print(f"  perf_pca: {df_perf_pca.shape}, methods: {sorted(df_perf_pca['method'].unique())}")
print(f"  pred_ridge: {df_pred_ridge.shape},  pred_pca: {df_pred_pca.shape}")
print(f"  ref pooled external R²:")
for k, v in ref_pooled_external.items():
    print(f"    {CLOCK_LABELS[k]:13s} {v:+.3f}")


# ── Diagnostic: which N does the "Network Clock" (Ridge) box in (f) use? ──
# per_sample_predictions_ridge_ext_best.csv carries predictions at the M that
# was externally-best for Ridge random-per-module — look it up here so it can
# be reported / cited in the paper.
ridge_clock_N = None
ridge_perf = _try_csv(TEST_DIR / "ridge_clock_perf_by_M.csv")
if ridge_perf is not None:
    pooled_ridge = ridge_perf[
        (ridge_perf["method"].astype(str).str.contains("random_per_cluster_ridge", case=False)) &
        (ridge_perf["scope"] == "pooled")
    ]
    if pooled_ridge.empty:
        # Fallback: any ridge-flavoured method, pooled
        pooled_ridge = ridge_perf[
            (ridge_perf["method"].astype(str).str.contains("ridge", case=False)) &
            (ridge_perf["scope"] == "pooled")]
    if not pooled_ridge.empty:
        ridge_clock_N = int(pooled_ridge.loc[pooled_ridge["r2"].idxmax(), "M"])
        print(f"\n  → Network Clock (Ridge random-per-module) external-best N = {ridge_clock_N} CpGs")
    else:
        print(f"\n  ridge_clock_perf_by_M.csv loaded but no pooled Ridge rows matched")
else:
    print(f"\n  ridge_clock_perf_by_M.csv not found — cannot report the Network Clock N here")


## Helpers

In [ ]:
def draw_fit_line(ax, x, y, color="black", lw=2.2, alpha=1.0):
    """Solid least-squares fit y ~ x, drawn across the data range, on top."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if len(x) < 2: return None, None
    m, b = np.polyfit(x, y, 1)
    xl = np.linspace(x.min(), x.max(), 100)
    ax.plot(xl, m*xl + b, linestyle="solid", color=color, lw=lw,
            alpha=alpha, zorder=4)
    return m, b


def square_panel(ax):
    """1:1 axes box so all six panels are the same visual size."""
    ax.set_box_aspect(1.0)


def panel_letter(ax, letter):
    ax.text(-0.18, 1.06, f"({letter})", transform=ax.transAxes,
            fontsize=22, fontweight="bold", va="bottom", ha="left")


def _save(fig, basename):
    p = OUT_DIR / basename
    fig.savefig(str(p) + ".png", dpi=DPI)
    fig.savefig(str(p) + ".pdf")
    print(f"  -> {p}.png / .pdf")


def draw_baselines_with_arrows(ax, x_data_max, ref_rows,
                                seg_x_frac=0.82, label_x_frac=1.03,
                                xlim_pad_frac=1.20,
                                y_label_band=(0.50, 0.95),
                                label_fontsize=10):
    """Draw dashed horizontal segments on the right portion of the plot and
    place each clock name with a short leader to its segment. Labels stagger
    vertically (descending by R²) so they never overlap.

    y_label_band : (lo, hi) fractions of the *current* ylim — labels are
        distributed between ymin + lo·(ymax-ymin) and ymin + hi·(ymax-ymin).
    """
    seg_x_start = seg_x_frac * x_data_max
    seg_x_end   = x_data_max
    label_x     = label_x_frac * x_data_max
    ax.set_xlim(0, xlim_pad_frac * x_data_max)

    for key, r2 in ref_rows:
        ax.hlines(r2, seg_x_start, seg_x_end,
                  colors=CLOCK_COLORS[key], linestyles="--", lw=1.3, alpha=0.95)

    # Top label = highest R²; labels span (lo, hi) of the ylim range
    sorted_rows = sorted(ref_rows, key=lambda r: -r[1])
    n = len(sorted_rows)
    y_lo, y_hi = y_label_band
    ymin, ymax = ax.get_ylim()
    y_labels = np.linspace(ymin + (ymax - ymin) * y_hi,
                           ymin + (ymax - ymin) * y_lo, n)

    for (key, r2), y_lab in zip(sorted_rows, y_labels):
        ax.annotate(
            CLOCK_LABELS[key],
            xy=(seg_x_end, r2), xycoords="data",
            xytext=(label_x, y_lab), textcoords="data",
            fontsize=label_fontsize, fontweight="bold",
            color=CLOCK_COLORS[key], va="center", ha="left",
            arrowprops=dict(arrowstyle="-", color=CLOCK_COLORS[key],
                            lw=0.9, shrinkA=1, shrinkB=1,
                            connectionstyle="arc3,rad=0"),
            annotation_clip=False,
        )


## Panel (a) — Test $R^{2}$ vs M, arrows to baseline labels

In [ ]:
def make_panel_a(ax):
    # Sweep curves (mean ± std across reps)
    agg = (df_sweep.groupby(["method", "n_modules"])["r2_test"]
                    .agg(["mean", "std"]).reset_index())
    for method, color in METHOD_COLOR.items():
        sub = agg[agg["method"] == method].sort_values("n_modules")
        if sub.empty: continue
        ax.errorbar(sub["n_modules"], sub["mean"], yerr=sub["std"].fillna(0),
                    fmt="o-", color=color, lw=1.8, markersize=6, capsize=3,
                    label=METHOD_LABEL[method])

    # Reference baselines from the training-test fold summary
    pretty_to_key = {v: k for k, v in CLOCK_LABELS.items()}
    ref_rows = []
    for _, row in df_sum.iterrows():
        cl = str(row.get("clock", ""))
        key = cl if cl in CLOCK_COLORS else pretty_to_key.get(cl)
        if key in REFERENCE_KEYS:
            ref_rows.append((key, float(row["r2_test"])))

    x_max = float(agg["n_modules"].max())
    ax.set_ylim(0.15, 0.95)
    draw_baselines_with_arrows(ax, x_max, ref_rows)

    ax.set_xlabel("Number of CpGs (N)")
    ax.set_ylabel(r"Test $R^{2}$")
    ax.legend(loc="lower left", handlelength=2.0)
    square_panel(ax)


fig, ax = plt.subplots(figsize=(PANEL_SIZE, PANEL_SIZE))
make_panel_a(ax); fig.tight_layout()
_save(fig, "panel_a_r2_vs_M")
plt.show()


## Panel (b) — Held-out test scatter at M = 100 (Ridge)

In [ ]:
def make_panel_b(ax):
    df = df_scatter_internal[df_scatter_internal["split"] == "test"]
    studies = sorted(df["study"].unique())
    color_for_study = {s: STUDY_PALETTE[i % len(STUDY_PALETTE)]
                       for i, s in enumerate(studies)}
    for s in studies:
        sub = df[df["study"] == s]
        ax.scatter(sub["y_true"], sub["y_pred"], s=40, alpha=0.85,
                   color=color_for_study[s], edgecolor="white", linewidth=0.4,
                   label=f"{s}", zorder=2)
    draw_fit_line(ax, df["y_true"], df["y_pred"])

    lims = [min(df["y_true"].min(), df["y_pred"].min()) - 2,
            max(df["y_true"].max(), df["y_pred"].max()) + 2]
    ax.set_xlim(lims); ax.set_ylim(lims)
    r2 = r2_score(df["y_true"], df["y_pred"])
    ax.set_xlabel("Chronological age (years)")
    ax.set_ylabel("Predicted age (years)")
    ax.set_title(f"(b)  $N$ = 100, $R^{{2}}$ = {r2:.3f}", loc="left")
    ax.legend(loc="upper left", fontsize=8, ncol=2, columnspacing=0.8, handletextpad=0.3)
    square_panel(ax)


fig, ax = plt.subplots(figsize=(PANEL_SIZE, PANEL_SIZE))
make_panel_b(ax); fig.tight_layout()
_save(fig, "panel_b_scatter_N100")
plt.show()


## Panel (c) — External-cohort scatter, Ridge clock

In [ ]:
def _make_ext_scatter(ax, df, age_col, pred_col, letter):
    if "group" not in df.columns:   df["group"]   = "unknown"
    if "dataset" not in df.columns: df["dataset"] = "unknown"
    groups = sorted(df["group"].unique(),
                    key=lambda g: (g not in ("Control", "Aging study"), g))
    datasets_in_data = list(df["dataset"].unique())
    fallback = ["o", "s", "^", "D", "v", "P", "X", "*"]
    for g in groups:
        sub_g = df[df["group"] == g]
        for i_d, dset in enumerate(datasets_in_data):
            sub = sub_g[sub_g["dataset"] == dset]
            if len(sub) == 0: continue
            marker = DATASET_MARKERS.get(dset, fallback[i_d % len(fallback)])
            ax.scatter(sub[age_col], sub[pred_col], s=55, marker=marker,
                       color=GROUP_COLORS.get(g, GROUP_COLORS["unknown"]),
                       edgecolor="white", linewidth=0.5, alpha=0.85, zorder=2)
    draw_fit_line(ax, df[age_col], df[pred_col])
    lims = [df[age_col].min() - 5, max(df[age_col].max(), df[pred_col].max()) + 5]
    ax.set_xlim(lims); ax.set_ylim(lims)
    r2  = r2_score(df[age_col], df[pred_col])
    ax.set_xlabel("Chronological age (years)")
    ax.set_ylabel("Predicted age (years)")
    ax.set_title(f"({letter})  $R^{{2}}$ = {r2:.3f}", loc="left")

    group_handles = [Line2D([0], [0], marker="o", linestyle="",
                            markerfacecolor=GROUP_COLORS.get(g, GROUP_COLORS["unknown"]),
                            markeredgecolor="white", markersize=9,
                            label=f"{g}")
                     for g in groups]
    leg_g = ax.legend(handles=group_handles, loc="upper left",
                      title="Disease group", title_fontsize=9,  fontsize=8)
    ax.add_artist(leg_g)
    dset_handles = [Line2D([0], [0], marker=DATASET_MARKERS.get(d, fallback[i % len(fallback)]),
                           linestyle="", markerfacecolor="#999",
                           markeredgecolor="white", markersize=9, label=str(d))
                    for i, d in enumerate(datasets_in_data)]
    ax.legend(handles=dset_handles, loc="lower right",
              title="Dataset", title_fontsize=9,  fontsize=8)
    square_panel(ax)


def _resolve_age_pred(df):
    age_col  = next((c for c in ["age", "Age", "chronological_age"]
                     if c in df.columns), None)
    pred_col = next((c for c in ["pred", "predicted", "predicted_age", "y_pred"]
                     if c in df.columns), None)
    if age_col is None or pred_col is None:
        raise KeyError(f"need age + predicted columns; have {list(df.columns)}")
    return age_col, pred_col


def make_panel_c(ax):
    df = df_scatter_external_ridge.copy()
    a, p = _resolve_age_pred(df)
    _make_ext_scatter(ax, df, a, p, "c")


fig, ax = plt.subplots(figsize=(PANEL_SIZE, PANEL_SIZE))
make_panel_c(ax); fig.tight_layout()
_save(fig, "panel_c_external_scatter_ridge")
plt.show()


## Panel (d) — PCA $R^{2}$ vs K on external cohorts, arrows to baselines

Same arrow style as (a). The PCA curve uses the cluster/module-PCA method (`method = "pca"`); baselines are the five reference clocks' **external-pooled R²** (computed in Cell 3 from `per_sample_predictions_pca_ext_best.csv`).

In [ ]:
def make_panel_d(ax):
    # Three PCA variants — modules / whole network / whole dataset
    pca_curves = [
        ("pca",         df_perf_pca,         "o"),
        ("pca_network", df_perf_pca_network, "s"),
        ("pca_array",   df_perf_pca_array,   "^"),
    ]
    x_max_global = 0.0
    for key, df_p, marker in pca_curves:
        if df_p is None or df_p.empty: continue
        sub = df_p[(df_p["method"] == key) & (df_p["scope"] == "pooled")].sort_values("M")
        if sub.empty: continue
        ax.plot(sub["M"], sub["r2"], f"{marker}-",
                color=CLOCK_COLORS[key], lw=1.8, markersize=6,
                label=PCA_SHORT_LABELS[key])
        x_max_global = max(x_max_global, float(sub["M"].max()))

    # Reference baselines — external pooled R²
    ref_rows = [(k, v) for k, v in ref_pooled_external.items() if k in REFERENCE_KEYS]
    ax.set_ylim(0.15, 0.95)
    if x_max_global > 0 and ref_rows:
        draw_baselines_with_arrows(ax, x_max_global, ref_rows)

    ax.set_xlabel("Number of PCs (K)")
    ax.set_ylabel(r"Pooled external $R^{2}$")
    ax.legend(title="PCA clocks", loc="lower right", handlelength=2.0,
              fontsize=9, title_fontsize=10)
    square_panel(ax)


fig, ax = plt.subplots(figsize=(PANEL_SIZE, PANEL_SIZE))
make_panel_d(ax); fig.tight_layout()
_save(fig, "panel_d_pca_r2_vs_K")
plt.show()


## Panel (e) — External-cohort scatter, PCA clock

In [ ]:
def make_panel_e(ax):
    df = df_scatter_external_pca.copy()
    a, p = _resolve_age_pred(df)
    _make_ext_scatter(ax, df, a, p, "e")


fig, ax = plt.subplots(figsize=(PANEL_SIZE, PANEL_SIZE))
make_panel_e(ax); fig.tight_layout()
_save(fig, "panel_e_external_scatter_pca")
plt.show()


## Panel (f) — Pooled MAE box plot: PCA + Ridge + 5 reference clocks

Order set by `BOX_ORDER` (Cell 1). Boxes use the global `CLOCK_COLORS`, so every clock looks identical to its appearance in (a) and (d).

In [ ]:
def _build_box_table():
    """Combine ridge and PCA per-sample predictions into one table with
    abs_err_<key> columns for every clock in BOX_ORDER (plus REFERENCE_KEYS).
    Both source CSVs use 'network' as their primary-clock column — we rename
    each to a clear key so they coexist."""
    age_r = _age_col(df_pred_ridge); age_p = _age_col(df_pred_pca)
    if age_r is None or age_p is None:
        raise KeyError("Missing age column in one of the per-sample CSVs.")

    dfr = df_pred_ridge.rename(columns={"network": "ridge"})
    dfp = df_pred_pca.rename(columns={"network": "pca"})

    # Anchor on PCA (has refs too); pull ridge predictions in by (dataset, gsm)
    merged = dfp.merge(dfr[["dataset", "gsm", "ridge"]],
                       on=["dataset", "gsm"], how="left")
    age_col = "age" if "age" in merged.columns else "Age"
    for clk in ["pca", "ridge"] + REFERENCE_KEYS:
        if f"abs_err_{clk}" not in merged.columns and clk in merged.columns:
            merged[f"abs_err_{clk}"] = (merged[clk] - merged[age_col]).abs()
    return merged


def make_panel_f(ax):
    df = _build_box_table()

    # Candidate keys: PCA on modules, Ridge Network Clock, and the 5 reference clocks
    candidates = ["pca", "ridge"] + REFERENCE_KEYS
    per_clock = {}
    for clk in candidates:
        col = f"abs_err_{clk}"
        if col not in df.columns:
            print(f"  (f) skipping {clk} — column {col} not in merged data")
            continue
        v = df[col].dropna().values
        if len(v): per_clock[clk] = v

    if BOX_ORDER is None:
        # Auto-sort ascending by mean MAE (best on the left)
        order = sorted(per_clock, key=lambda k: per_clock[k].mean())
    else:
        order = [k for k in BOX_ORDER if k in per_clock]
        order += [k for k in per_clock if k not in order]

    data_for_boxes = [per_clock[k] for k in order]
    mae_per_box    = [v.mean() for v in data_for_boxes]
    box_colors     = [CLOCK_COLORS[k] for k in order]
    xtick_labels   = [CLOCK_LABELS[k] for k in order]

    positions = np.arange(len(data_for_boxes))
    bp = ax.boxplot(data_for_boxes, positions=positions, widths=0.6,
                    patch_artist=True, showfliers=False,
                    medianprops={"color": "black", "linewidth": 1.6},
                    whiskerprops={"color": "#222", "linewidth": 1.1},
                    capprops={"color": "#222", "linewidth": 1.1})
    for patch, c in zip(bp["boxes"], box_colors):
        patch.set_facecolor(c); patch.set_alpha(0.55); patch.set_edgecolor("black")

    # Strip-plot overlay, coloured by group when available
    rng = np.random.RandomState(0)
    group_col = "group" if "group" in df.columns else None
    for i, clk in enumerate(order):
        err = f"abs_err_{clk}"
        cols_needed = [err] + ([group_col] if group_col else [])
        sub = df[cols_needed].dropna()
        if group_col:
            pt = [GROUP_COLORS.get(g, GROUP_COLORS["unknown"]) for g in sub[group_col]]
        else:
            pt = "#444"
        x = np.full(len(sub), i) + rng.uniform(-0.18, 0.18, len(sub))
        ax.scatter(x, sub[err].values, s=12, c=pt, edgecolor="black",
                   linewidth=0.2, alpha=0.55, zorder=3)

    y_top = max(v.max() for v in data_for_boxes) * 1.07
    for i, m in enumerate(mae_per_box):
        ax.text(i, y_top, f"MAE\n{m:.2f}", ha="center", va="bottom",
                fontsize=10, fontweight="bold")
    ax.set_xticks(positions)
    ax.set_xticklabels(xtick_labels, fontsize=12, rotation=20, ha="right")
    ax.set_ylabel("Per-sample |error| (years)")
    ax.set_ylim(0, y_top * 1.22)
    square_panel(ax)


fig, ax = plt.subplots(figsize=(PANEL_SIZE, PANEL_SIZE))
make_panel_f(ax); fig.tight_layout()
_save(fig, "panel_f_boxplot_combined")
plt.show()


## Composite figure — all six panels (2 × 3)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=COMPOSITE_FIGSIZE)
make_panel_a(axes[0, 0]); panel_letter(axes[0, 0], "a")
make_panel_b(axes[0, 1])
make_panel_c(axes[0, 2])
make_panel_d(axes[1, 0]); panel_letter(axes[1, 0], "d")
make_panel_e(axes[1, 1])
make_panel_f(axes[1, 2]); panel_letter(axes[1, 2], "f")
fig.tight_layout()
_save(fig, "composite_figure")
fig.savefig(str(OUT_DIR / "composite_figure.svg"))
print(f"  -> {OUT_DIR / 'composite_figure.svg'}")
plt.show()
